# Experiment 0: Baseline Topic Modeling

This notebook runs a simple rule-based keyword matching baseline for topic modeling, using the same preprocessed input as the LDA and NMF experiments.

## Hypothesis:
The baseline is built on the following core assumptions:

H1 - Rule-based keyword matching is sufficient to capture clear, explicit topics in customer service tickets.
We assume that tickets mentioning specific keywords (e.g., "login", "network", "crash") can be reliably mapped to predefined topic categories.

H2 - The performance of this baseline will serve as a lower bound for the more complex LDA and NMF models.
If LDA/NMF do not outperform this simple approach, they do not provide meaningful improvements for this dataset.

H3 - Explicit keywords dominate implicit thematic meaning in this customer service domain.
Tickets are expected to contain direct mentions of the problem type, making keyword matching a reasonable baseline.

## Pipeline:
**load data → define topic keywords → match rules → assign topics → export results**

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
PROJECT_ROOT = Path.cwd().parents[2]
axis2_dir = PROJECT_ROOT / 'notebooks' / '03_Topic_and_Insights'
sys.path.insert(0, str(axis2_dir))
from Data_preprocessing import Parameters_Path as config

In [3]:
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'Experiment_4_Baseline'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_TOPICS_PATH = RESULTS_DIR / 'Experiment_4_baseline_topics.json'
DOC_TOPIC_PATH = RESULTS_DIR / 'Experiment_4_baseline_doc_topics.csv'

tfidf_cfg = config.PARAMETERS['tfidf_vectorizer']
base_cfg = config.PARAMETERS['nmf']
N_COMPONENTS = int(base_cfg['n_components'])
N_TOP_WORDS = int(base_cfg['n_top_words'])
RANDOM_STATE = int(base_cfg['random_state'])

In [4]:
df = pd.read_csv(INPUT_PATH)
texts = df['text_cleaned_axis1'].astype(str).str.strip()
texts = texts[texts.ne('')]

# Baseline: TF-IDF + lexical clustering baseline

In [5]:
vectorizer = TfidfVectorizer(
    max_df=tfidf_cfg['max_df'],
    min_df=tfidf_cfg['min_df'],
    max_features=tfidf_cfg['max_features'],
    stop_words=tfidf_cfg['stop_words']
)
X_tfidf = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

# Simple baseline: random lexical grouping + mean feature weights
np.random.seed(RANDOM_STATE)
base_components = np.random.rand(N_COMPONENTS, X_tfidf.shape[1])
for i in range(N_COMPONENTS):
    base_components[i] = X_tfidf[np.random.choice(X_tfidf.shape[0], 200)].mean(axis=0)

doc_topic = X_tfidf @ base_components.T

In [6]:
def extract_baseline_topics(components, vocab, n_top_words=8):
    topics = {}
    for i, comp in enumerate(components, start=1):
        idx = np.argsort(comp)[-n_top_words:][::-1]
        topics[f'topic_{i}'] = {
            'keywords': [vocab[j] for j in idx],
            'weights': comp[idx].tolist()
        }
    return topics

base_topics = extract_baseline_topics(base_components, feature_names, N_TOP_WORDS)

with open(BASE_TOPICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(base_topics, f, indent=2, ensure_ascii=False)

# Normalize for confidence comparison
doc_topic_norm = np.abs(doc_topic) / (np.max(np.abs(doc_topic), axis=1, keepdims=True) + 1e-8)
topic_cols = [f'base_topic_{i}' for i in range(1, N_COMPONENTS + 1)]
doc_df = pd.DataFrame(doc_topic_norm, columns=topic_cols)
doc_df['baseline_dominant_topic'] = doc_df[topic_cols].values.argmax(axis=1) + 1
doc_df['max_topic_prob'] = doc_df[topic_cols].max(axis=1)
doc_df.to_csv(DOC_TOPIC_PATH, index=False, encoding='utf-8')

In [7]:
print("Baseline Topic Keywords:")
for t_name, content in base_topics.items():
    print(f"{t_name}: {', '.join(content['keywords'])}")

Baseline Topic Keywords:
topic_1: inquiry, refund, account, cancellation, software, technical, data, compatibility
topic_2: inquiry, refund, account, software, data, resolve, cancellation, billing
topic_3: account, inquiry, software, refund, billing, cancellation, data, access
topic_4: technical, cancellation, data, software, refund, account, inquiry, started
